# Ejecutar Proyecto ADSO en Google Colab

Este notebook inicia temporalmente el proyecto PHP con MariaDB en el entorno de Colab. Los datos se borran cuando termina la sesión; no lo uses con información real. Ejecuta las celdas en orden y permite el acceso a internet en el entorno.

El repositorio debe ser público para clonarlo con la celda siguiente. Si es privado, sube una copia ZIP del proyecto a Colab, extráela en `/content/ProyectoAdso` y luego ejecuta las celdas desde la instalación.

In [ ]:
import os
import subprocess

subprocess.run(
    ["bash", "-lc", "apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq php-cli php-mysql mariadb-server git"],
    check=True,
)
print("PHP, PDO MySQL y MariaDB instalados.")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/ProyectoAdso")
if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/perezrinconsantiesteban-dot/ProyectoAdso.git", str(PROJECT_DIR)],
        check=True,
    )

assert (PROJECT_DIR / "Config" / "bdClub.sql").is_file(), "No se encontró Config/bdClub.sql en el proyecto."
print(f"Proyecto listo en {PROJECT_DIR}")

In [ ]:
import secrets

subprocess.run(["service", "mariadb", "start"], check=True)
subprocess.run(["mariadb-admin", "ping", "--silent"], check=True)

with (PROJECT_DIR / "Config" / "bdClub.sql").open("rb") as schema_file:
    subprocess.run(["mariadb"], stdin=schema_file, check=True)

DB_PASSWORD = secrets.token_hex(24)
db_setup = (
    "CREATE USER IF NOT EXISTS 'club_app'@'127.0.0.1' IDENTIFIED BY '" + DB_PASSWORD + "';\n"
    "ALTER USER 'club_app'@'127.0.0.1' IDENTIFIED BY '" + DB_PASSWORD + "';\n"
    "GRANT ALL PRIVILEGES ON bdClub.* TO 'club_app'@'127.0.0.1';\n"
    "FLUSH PRIVILEGES;\n"
)
subprocess.run(["mariadb"], input=db_setup, text=True, check=True)
print("Base de datos bdClub creada y usuario temporal configurado.")

In [ ]:
import time

APP_ENV = os.environ.copy()
APP_ENV.update({
    "DB_HOST": "127.0.0.1",
    "DB_NAME": "bdClub",
    "DB_USER": "club_app",
    "DB_PASSWORD": DB_PASSWORD,
})

php_check = subprocess.run(
    ["php", "-r", "$pdo = new PDO('mysql:host=127.0.0.1;dbname=bdClub;charset=utf8mb4', getenv('DB_USER'), getenv('DB_PASSWORD')); echo 'Conexión PHP-MySQL correcta';"],
    env=APP_ENV,
    capture_output=True,
    text=True,
    check=True,
)
print(php_check.stdout)

In [ ]:
# La página inicial redirige al formulario de acceso.
(PROJECT_DIR / "index.php").write_text(
    "<?php header('Location: /Vista/Login.html'); exit;\n",
    encoding="utf-8",
)
server_log = open("/tmp/proyectoadso-php.log", "w")
server_process = subprocess.Popen(
    ["php", "-S", "0.0.0.0:8000", "-t", str(PROJECT_DIR)],
    cwd=PROJECT_DIR,
    env=APP_ENV,
    stdout=server_log,
    stderr=subprocess.STDOUT,
)
time.sleep(2)
if server_process.poll() is not None:
    server_log.close()
    raise RuntimeError(Path("/tmp/proyectoadso-php.log").read_text(encoding="utf-8"))
print("Servidor PHP iniciado en el puerto 8000.")

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8000, height=700)

La página de acceso se abre en el recuadro. Registra primero un asociado para probar el inicio de sesión. Esta configuración permite probar el registro e inicio de sesión; los módulos de préstamos, pagos y actividades aún no tienen conexión a base de datos en el código actual.